# RBI Financial Stability Report (June 2026) – RAG Chatbot

A local Retrieval-Augmented Generation (RAG) chatbot that answers questions from the RBI Financial Stability Report, June 2026, and shows the PDF pages each answer came from.

**Stack:** Python, LangChain, ChromaDB, Hugging Face embeddings (all-MiniLM-L6-v2), Ollama (qwen3:1.7b)

**Source document:** https://www.rbi.org.in/ScriptS/FsReports.aspx

## 1. Load the PDF

In [1]:
from langchain_community.document_loaders import PyPDFLoader

PDF_PATH = "data/RBI's Financial Stability Report 2026.pdf"

loader = PyPDFLoader(PDF_PATH)
pages = loader.load()

print("Pages loaded:", len(pages))
print(pages[10].metadata)
print(pages[10].page_content[:500])

C:\Users\finou\AppData\Local\Temp\ipykernel_18060\1796133470.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Pages loaded: 193
{'producer': 'Adobe Acrobat Pro (32-bit) 24.4.20243', 'creator': 'Adobe Acrobat Pro (32-bit) 24.4.20243', 'creationdate': '2026-06-30T15:56:29+05:30', 'moddate': '2026-07-06T16:51:55+05:30', 'source': "data/RBI's Financial Stability Report 2026.pdf", 'total_pages': 193, 'page': 10, 'page_label': '11'}
Page No.
Financial Stability Report June 2026
1.37 Sound Financials Strengthen the Banking System 32
1.38 Improving Asset Quality and Credit Cost Driving Credit Growth 33
1.39 Evolving Liability Profile and Impact on Cost of Funds 33
1.40 Impact of Waning Deposit Franchise on Profitability and Liquidity 34
1.41 Banks Softened their Margin Pressure Despite Decline in CASA Share 35
1.42 Strong Credit Growth Alongside Benign Asset Quality 35
1.43 Microfinance Sector – Recovery in Credit Amid Fallin


## 2. Remove front matter and split into chunks

PDF pages 7–18 are the contents, list of charts, list of tables and abbreviations. They add noise to retrieval, so they are excluded. The text is then split into overlapping chunks of about 1000 characters.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# index 6-17 = PDF pages 7-18
EXCLUDE = set(range(6, 18))
pages_clean = [p for p in pages if p.metadata["page"] not in EXCLUDE]
print("Pages kept:", len(pages_clean))

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(pages_clean)

# drop very short chunks (mostly headers and chart labels)
chunks = [c for c in chunks if len(c.page_content.strip()) > 100]
print("Chunks:", len(chunks))

Pages kept: 181
Chunks: 609


## 3. Create embeddings and the vector store

Each chunk is converted to a vector and stored in a local Chroma database. If the database folder already exists, it is loaded instead of rebuilt. Delete the `chroma_db` folder if you change the chunking settings.

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

DB_DIR = "chroma_db"
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(DB_DIR):
    vectorstore = Chroma(persist_directory=DB_DIR, embedding_function=embeddings)
    print("Loaded existing database")
else:
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_DIR
    )
    print("Created new database")

print("Chunks in database:", vectorstore._collection.count())

## 4. Test retrieval

Check which chunks are retrieved for a question before adding the language model.

In [4]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
docs = retriever.invoke("What are the key risks to the Indian financial system?")

for d in docs:
    print("PDF page:", d.metadata["page_label"])
    print(d.page_content[:300])
    print("-----")

PDF page: 19
1
Overview
1 This report reflects information available as of June 10, 2026, unless explicitly specified otherwise.
The Financial Stability Report (FSR) is a half-
yearly publication, with contributions from all 
financial sector regulators. It presents the collective 
assessment of the Sub Committe
-----
PDF page: 5
healthy profitability, low levels of non-performing assets, and robust credit growth. Stress tests indicate that 
the financial institutions are well-positioned to withstand adverse shocks. Financial markets, notwithstanding 
the volatility posed by global uncertainties, have been functioning in an 
-----
PDF page: 41
23
Financial Stability Report June 2026
yields, has led banks to increase their holdings 
of these securities (Chart 1.24 a and b). At the 
same time, long-term institutional investors such 
as insurers and pension funds have gradually 
diversified a larger share of their portfolios towards 
equitie
-----


## 5. Build the question-answering function

The top matching chunks are passed to a local Qwen3 model through Ollama. The prompt tells the model to answer only from the retrieved context, and to say so when the answer is not in the report. Source pages are printed with each answer.

Make sure Ollama is running and `qwen3:1.7b` is pulled.

In [5]:
import re
from langchain_ollama import OllamaLLM
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = OllamaLLM(model="qwen3:1.7b", num_ctx=4096, temperature=0)

prompt = ChatPromptTemplate.from_template(
    """You are an assistant answering questions about the RBI Financial Stability Report (June 2026).
Answer ONLY using the context below. If the answer is not in the context,
say: "I couldn't find this in the report."
Answer in 2 to 4 sentences.

Context:
{context}

Question: {question}

Answer:"""
)

chain = prompt | llm | StrOutputParser()

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

def ask(question, k=3):
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)
    answer = chain.invoke({"context": context, "question": question})
    # remove any <think>...</think> block that Qwen3 may print
    answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()
    pages_used = sorted({d.metadata["page_label"] for d in docs}, key=int)

    print("Q:", question)
    print("A:", answer)
    if "couldn't find this in the report" not in answer:
        print("Sources (PDF pages):", ", ".join(pages_used))
    print("-" * 60)

## 6. Example questions

In [ ]:
ask("What does the report say about NBFC asset quality?")
ask("What does the report say about cyber risk?")
ask("What are the risks from rising oil prices?")
ask("What are the risks related to artificial intelligence?", k=5)

Q: What does the report say about NBFC asset quality?
A: The report highlights improved asset quality for NBFCs, with a steady decline in GNPA ratios for both upper (NBFC-UL) and middle (NBFC-ML) layers, reaching a low of 0.8% net NPA. Slippage ratios moderated, particularly for middle-layer NBFCs, while upper-layer NBFCs remain slightly elevated but show early stabilization. Overall, asset quality improved across sectors, reflecting stronger credit quality and reduced non-performing assets.
Sources (PDF pages): 57, 58, 107
------------------------------------------------------------
Q: What does the report say about cyber risk?
A: The report highlights that cyber risk exposure is generally perceived as manageable by institutions, with 98% rating risk levels as very low to moderate. However, there is a moderate or significant increase in cyber risk compared to 2025–26, reflecting the evolving threat landscape. The report underscores the need for enhanced AI-enabled threat preparedness,

## 7. Out-of-scope question

The chatbot should refuse to answer questions that are not covered by the report.

In [7]:
ask("Who won the 2022 FIFA World Cup?")

Q: Who won the 2022 FIFA World Cup?
A: I couldn't find this in the report.
------------------------------------------------------------


## 8. Known limitation

Small models can mix up details. For the question below, the answer was mostly right (the 22.7 lakh decline, the 5.9% growth, and the role of NBFCs), but it attached "first rise in seven quarters" to credit, when the report says credit increased and it was the active borrower base that fell for the first time in seven quarters. Always check answers against the cited pages.

In [ ]:
ask("What is said about the microfinance sector?")

## Limitations

- Uses a small 1.7B-parameter model so it can run on a laptop with 8 GB RAM, so answers can be brief or occasionally wrong.
- In a manual check of 6 questions against the report, 5 were fully correct and 1 had a misattributed detail (see section 8). This is a small sample, not a general accuracy figure.
- Numbers from tables and charts can be extracted poorly or attached to the wrong metric.
- Single document only; no chat history.
- Answers may contain errors, so verify them against the cited PDF pages.

## Possible improvements

- Try a larger model, if hardware allows.
- Support multiple editions of the report and filter by edition.
- Add a reranker or hybrid (BM25 + embeddings) search.
- Evaluate retrieval and answers on a larger set of test questions, possibly graded by a stronger model.
- Add a simple Streamlit interface.